In [1]:
import torch
import argparse
import os
from omegaconf import OmegaConf
import numpy as np

# 
from utils.logging import log_string
from utils.data_loader import load_dataset # 
from utils.spatial import HierarchicalKDTreePartitioner
from models.stcp_net import STCPNet
from trainer import STCPTrainer

config = OmegaConf.load("configs/stcp_lorenz.yaml")
if config.data.dataset_type == 'synthetic':
    log_name = f"STCP_Synthetic_{config.synthetic.data_type}_{config.synthetic.n_nodes}N.log"
else:
# 
    data_name = os.path.basename(config.data.traffic_file).split('.')[0]
    log_name = f"STCP_Real_{data_name}.log"
log_file_path = os.path.join(config.log_dir, log_name)
log_f = open(log_file_path, 'w') # 
config

{'project_name': 'STCP_Lorenz_Test', 'log_dir': './logs', 'model_save_path': './saved_models/stcp_lorenz_best.pth', 'mode': 'train', 'data': {'dataset_type': 'synthetic', 'input_len': 10, 'output_len': 1, 'train_ratio': 0.6, 'val_ratio': 0.2, 'test_ratio': 0.2, 'input_dim': 1}, 'synthetic': {'data_type': 'lorenz', 'n_nodes': 128, 'num_steps_after_burn_in': 5000, 'num_steps_burn_in': 1000, 'var_p': 2, 'spatial_layout': 'ring'}, 'spatial': {'initial_depth': 1, 'final_depth': 7, 'schedule_type': 'linear'}, 'model': {'n_nodes': 20, 'embed_dim': 32, 'use_spatial_emb': True, 'use_temporal_emb': False, 'node_emb_dim': 8, 'tod_size': 288, 'tod_emb_dim': 1, 'dow_size': 7, 'dow_emb_dim': 1, 'num_layers': 2, 'num_heads': 4, 'mlp_ratio': 4.0, 'dropout': 0.1, 'attn_dropout': 0.1}, 'training': {'seed': 42, 'device': 'cuda:1', 'batch_size': 64, 'max_epoch': 100, 'loss_null_val': 'None', 'lr_data_start': 0.001, 'weight_decay_data': 0.0001, 'lr_data_milestones': [50, 80], 'lr_data_gamma': 0.1, 'lr_grap

In [2]:
device = torch.device(config.training.device if torch.cuda.is_available() else "cpu")
log_string(log_f, f"Using device: {device}")
dataset_pack = load_dataset(config, log_f)

config.model.n_nodes = dataset_pack['n_nodes']
log_string(log_f, f"Number of nodes set to: {config.model.n_nodes}")

# 4. 
partitioner = None 
if dataset_pack['locations'] is not None:
    log_string(log_f, "Initializing KDTree Partitioner (spatial data found).")
    partitioner = HierarchicalKDTreePartitioner(
        locations=dataset_pack['locations'],
        config=config.spatial
    )
    # 
    if not config.model.use_spatial_emb:
            log_string(log_f, "Warning: Spatial data found, but use_spatial_emb=False in config.")
else:
    log_string(log_f, "Skipping Partitioner (no spatial data found).")
    # 
    if config.model.use_spatial_emb:
        config.model.use_spatial_emb = False
        log_string(log_f, "Warning: No spatial data found. Forcing use_spatial_emb = False.")


# 5. 
model = STCPNet(config).to(device)

# 6. 
trainer = STCPTrainer(
    config, model, partitioner, 
    dataset_pack, device, log_f
)

Using device: cuda:1
Generating Synthetic Causal Dataset...
Shape of generated data (after burn-in): (5000, 128, 1)
Shape of generated locations: (2, 128)
Data shapes (Samples, Time, Nodes, Features):
Shape of Train X (P=10): (2990, 10, 128, 1)
Shape of Train Y (Q=1): (2990, 1, 128, 1)
Shape of Validation X (P=10): (990, 10, 128, 1)
Shape of Validation Y (Q=1): (990, 1, 128, 1)
Shape of Test Y (P=10): (990, 10, 128, 1)
Shape of Test Y (Q=1): (990, 1, 128, 1)
Mean (Train X): 2.575074437075041 & Std (Train X): 4.37175954865589
Number of nodes set to: 128
Initializing KDTree Partitioner (spatial data found).
Using loss null_val: None
Seed set to 42


In [3]:
trainer.train()

======================TRAIN MODE======================
Epoch 1: Hierarchy changing. Depth: 1, Groups: 2
Epoch 001, Time: 2.0s | Loss_Data: 0.4798 | Loss_Graph: 0.5260 | Loss_Sparse: 0.5621
--- Validating at epoch 1 ---
---> VAL @ 1: MAE: 1.6067 | Causal ROC-AUC: 0.7494 | Causal PR-AUC: 0.5131 | Causal F1: 0.0462
---> Best model saved at epoch 1.
Epoch 002, Time: 1.7s | Loss_Data: 0.3477 | Loss_Graph: 0.3889 | Loss_Sparse: 0.4458
--- Validating at epoch 2 ---
---> VAL @ 2: MAE: 1.3554 | Causal ROC-AUC: 0.7494 | Causal PR-AUC: 0.5131 | Causal F1: 0.0000
Epoch 003, Time: 1.7s | Loss_Data: 0.3081 | Loss_Graph: 0.3393 | Loss_Sparse: 0.3391
--- Validating at epoch 3 ---
---> VAL @ 3: MAE: 1.2464 | Causal ROC-AUC: 0.7494 | Causal PR-AUC: 0.5131 | Causal F1: 0.0000
Epoch 004, Time: 1.7s | Loss_Data: 0.2865 | Loss_Graph: 0.3096 | Loss_Sparse: 0.2543
--- Validating at epoch 4 ---
---> VAL @ 4: MAE: 1.1907 | Causal ROC-AUC: 0.7494 | Causal PR-AUC: 0.5131 | Causal F1: 0.0000
Epoch 005, Time: 1.7s 